In [11]:
!pip install -q pydeseq2 gseapy

In [12]:
import argparse
import pathlib
import sys
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats


def make_demo_data(outdir):
    rng = np.random.default_rng(42)
    n_genes, n_per = 3000, 6
    genes = [f"GENE{i:04d}" for i in range(n_genes)]
    samples = [f"ctrl_{i+1}" for i in range(n_per)] + [f"treat_{i+1}" for i in range(n_per)]
    base = rng.gamma(2, 150, n_genes)
    fold = np.ones(n_genes)
    fold[rng.choice(n_genes, 300, replace=False)] = rng.choice([0.25, 0.4, 2.5, 4.0], 300)
    counts = np.zeros((n_genes, len(samples)), dtype=int)
    for j, s in enumerate(samples):
        mu = base * (fold if s.startswith("treat") else 1)
        counts[:, j] = rng.negative_binomial(10, 10 / (10 + mu))

    demo_dir = outdir / "demo_inputs"
    demo_dir.mkdir(parents=True, exist_ok=True)
    c = pd.DataFrame(counts, index=genes, columns=samples)
    m = pd.DataFrame({"condition": ["control"] * n_per + ["treated"] * n_per}, index=samples)
    c.to_csv(demo_dir / "demo_counts.csv")
    m.to_csv(demo_dir / "demo_metadata.csv")
    return c, m


def run_deseq(counts, meta, cond_col):
    try:
        dds = DeseqDataSet(counts=counts.T, metadata=meta, design=f"~{cond_col}")
    except TypeError:
        dds = DeseqDataSet(counts=counts.T, metadata=meta, design_factors=cond_col)
    dds.deseq2()
    return dds


def volcano(res, out, lfc_thr, padj_thr):
    df = res.dropna(subset=["padj"]).copy()
    df["nlog10"] = -np.log10(df["padj"].clip(lower=1e-300))
    df["status"] = "NS"
    df.loc[(df.padj < padj_thr) & (df.log2FoldChange >= lfc_thr), "status"] = "Up"
    df.loc[(df.padj < padj_thr) & (df.log2FoldChange <= -lfc_thr), "status"] = "Down"
    plt.figure(figsize=(7, 6))
    sns.scatterplot(
        data=df,
        x="log2FoldChange",
        y="nlog10",
        hue="status",
        palette={"Up": "#d62728", "Down": "#1f77b4", "NS": "#bbbbbb"},
        s=14,
        linewidth=0,
    )
    plt.axhline(-np.log10(padj_thr), ls="--", c="k", lw=0.7)
    for x in (lfc_thr, -lfc_thr):
        plt.axvline(x, ls="--", c="k", lw=0.7)
    plt.xlabel("log2 fold change")
    plt.ylabel("-log10 adj. p-value")
    plt.title("Volcano plot")
    plt.tight_layout()
    plt.savefig(out, dpi=200)
    plt.close()


def normed_log(dds):
    norm = pd.DataFrame(dds.layers["normed_counts"], index=dds.obs_names, columns=dds.var_names)
    return np.log2(norm + 1)


def heatmap(dds, res, out, top_n=30):
    top = res.dropna(subset=["padj"]).sort_values("padj").head(top_n).index
    mat = normed_log(dds)[top].T
    mat = mat.sub(mat.mean(axis=1), axis=0).div(mat.std(axis=1) + 1e-9, axis=0)
    g = sns.clustermap(mat, cmap="vlag", center=0, figsize=(8, 9))
    g.savefig(out, dpi=200)
    plt.close("all")


def pca_plot(dds, meta, cond_col, out):
    x = normed_log(dds)
    x = x - x.mean()
    u, s, _ = np.linalg.svd(x.values, full_matrices=False)
    pcs, var = u[:, :2] * s[:2], (s ** 2) / (s ** 2).sum()
    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=meta.loc[x.index, cond_col].values, s=80)
    plt.xlabel(f"PC1 ({var[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({var[1]*100:.1f}%)")
    plt.title("PCA of samples")
    plt.tight_layout()
    plt.savefig(out, dpi=200)
    plt.close()


def enrichment(genes, outdir):
    import gseapy as gp

    enr = gp.enrichr(
        gene_list=genes,
        organism="human",
        gene_sets=["GO_Biological_Process_2023", "KEGG_2021_Human"],
        outdir=None,
    )
    enr.results.to_csv(outdir / "enrichment.csv", index=False)
    top = enr.results.sort_values("Adjusted P-value").head(15)
    plt.figure(figsize=(8, 6))
    plt.barh(top["Term"][::-1], -np.log10(top["Adjusted P-value"][::-1]))
    plt.xlabel("-log10 adj. p")
    plt.title("Top enriched terms")
    plt.tight_layout()
    plt.savefig(outdir / "enrichment.png", dpi=200)
    plt.close()


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true", help="generate and analyse synthetic data")
    ap.add_argument("--counts")
    ap.add_argument("--metadata")
    ap.add_argument("--condition-col", default="condition")
    ap.add_argument("--treated", default="treated")
    ap.add_argument("--control", default="control")
    ap.add_argument("--lfc", type=float, default=1.0)
    ap.add_argument("--padj", type=float, default=0.05)
    ap.add_argument("--min-counts", type=int, default=10)
    ap.add_argument("--enrich", action="store_true", help="Enrichr (internet + real gene symbols)")
    ap.add_argument("--outdir", default="results")
    a = ap.parse_args(argv)

    out = pathlib.Path(a.outdir)
    out.mkdir(parents=True, exist_ok=True)
    if a.demo:
        counts, meta = make_demo_data(out)
    else:
        if not (a.counts and a.metadata):
            ap.error("provide --counts and --metadata, or use --demo")
        counts = pd.read_csv(a.counts, index_col=0)
        meta = pd.read_csv(a.metadata, index_col=0)

    common_samples = [s for s in meta.index if s in counts.columns]
    if len(common_samples) != len(meta.index):
        missing = set(meta.index) - set(common_samples)
        raise ValueError(f"Samples in metadata missing from counts: {sorted(missing)}")

    meta = meta.loc[common_samples]
    counts = counts[common_samples].dropna(how="all")
    counts = counts[counts.sum(axis=1) >= a.min_counts]
    print(f"{counts.shape[0]} genes x {counts.shape[1]} samples after filtering")

    dds = run_deseq(counts.round().astype(int), meta, a.condition_col)
    stat = DeseqStats(dds, contrast=[a.condition_col, a.treated, a.control])
    stat.summary()
    res = stat.results_df
    res.to_csv(out / "de_results.csv")

    sig = res[(res.padj < a.padj) & (res.log2FoldChange.abs() >= a.lfc)]
    print(f"Significant: {len(sig)} (up {(sig.log2FoldChange > 0).sum()}, down {(sig.log2FoldChange < 0).sum()})")

    volcano(res, out / "volcano.png", a.lfc, a.padj)
    heatmap(dds, res, out / "heatmap_top30.png")
    pca_plot(dds, meta, a.condition_col, out / "pca.png")
    if a.enrich and len(sig):
        enrichment(sig.index.tolist(), out)
    print("Done ->", out)


if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        main(["--demo"])
    else:
        main()

3000 genes x 12 samples after filtering
Log2 fold change & Wald test p-value: condition treated vs control
            baseMean  log2FoldChange     lfcSE      stat        pvalue  \
GENE0000  288.486607       -0.130863  0.259025 -0.505215  6.134076e-01   
GENE0001  456.595921       -0.020201  0.221347 -0.091263  9.272834e-01   
GENE0002  257.472453       -0.032512  0.290730 -0.111830  9.109581e-01   
GENE0003  261.542648       -0.291027  0.268025 -1.085821  2.775581e-01   
GENE0004  487.596294        0.467910  0.234135  1.998466  4.566613e-02   
...              ...             ...       ...       ...           ...   
GENE2995  233.555392        1.718233  0.195003  8.811311  1.236911e-18   
GENE2996  529.182755        0.092595  0.238156  0.388799  6.974247e-01   
GENE2997  452.441263        0.428804  0.283614  1.511929  1.305520e-01   
GENE2998  246.162726        0.416272  0.279253  1.490660  1.360508e-01   
GENE2999  236.009529       -0.340520  0.262908 -1.295207  1.952490e-01   

    